In [53]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
import plotly.express as px

def wrangle (filename, encoding = None, dropna_columnnames = None, change_column_name= None, date_column= None, remove_duplicated_rows_columnsname = None):
    # Loading Data from csv file
    df = pd.read_csv(filename, encoding= encoding)
    ### Get Orders NaN Rows
    df.dropna(subset= dropna_columnnames, inplace=True)

    # Rename Columns
    df = df.rename(columns= change_column_name)
    
    # Change OrderDate and ID Columns type to Date, integer
    df[date_column] = pd.to_datetime(df[date_column])    

    #Drop dupliacted rows
    df = df.drop_duplicates(subset= remove_duplicated_rows_columnsname)

       
    #Strip and lowercase columns names
    df.columns = df.columns.str.strip().str.lower()
    
    for col in df.columns:
        if 'id' in col or 'ordernumber' in col:
            df[col] = df[col].astype(int)
    return df 


orders = wrangle('all_data.csv', encoding= 'latin-1' , 
                 dropna_columnnames= ['OrderID'], 
                 change_column_name= {'City.1' : 'SuppCity', 'Country.1' : 'SuppCountry', 'Phone.1' : 'Supp_Phone'}, 
                 date_column= 'OrderDate', 
                 remove_duplicated_rows_columnsname= ['OrderID'])
orders = orders[['customerid', 'firstname', 'lastname', 'city', 'country', 'orderid', 'ordernumber', 'orderdate', 'totalamount']]


In [54]:
ordesr = orders[orders.totalamount < 15000]

# Bivariate Analysis

### country and tottal amount
### city and tottal  amount

In [55]:
def plots(columnname, top10= False):
    prop = orders.groupby(columnname, as_index=False)['totalamount'].sum().sort_values('totalamount',ascending=False)
    prop['proper'] = (prop.totalamount / orders.totalamount.sum()) * 100
    prop = prop.sort_values(by='proper', ascending=False)
    print("Describe of all Toatal amount valeus ", prop.totalamount.describe())
    if top10:
        prop = prop.head(10)
    fig1 = px.bar(data_frame=prop, x = columnname, y='totalamount')
    fig2 = px.pie(data_frame=prop, names=columnname, values='totalamount')
    fig1.show()
    fig2.show()


In [56]:
plots('country')

Describe of all Toatal amount valeus  count        21.000000
mean      64498.028095
std       72328.731450
min        3531.950000
25%       19431.890000
50%       35134.980000
75%       60814.890000
max      263566.980000
Name: totalamount, dtype: float64


In [57]:
plots('city', True)

Describe of all Toatal amount valeus  count        69.000000
mean      19629.834638
std       24804.361376
min         357.000000
25%        4788.060000
50%       11830.100000
75%       23850.950000
max      117483.390000
Name: totalamount, dtype: float64


## Date and totalamount

In [58]:
date_total = orders[['orderdate', 'totalamount']]
date_total = date_total.set_index('orderdate')
px.line(data_frame=date_total)

## Data and Country

In [ ]:
date_country = orders[['orderdate', 'country']]
date_country = date_country.set_index('orderdate')
date_country = date_country.country.resample(rule='1ME').nunique()

In [68]:
px.line(data_frame=date_country, y='country')

## Customers and Totalamount

In [75]:
cust_orders = {
    "fullname" : orders.firstname + " " + orders.lastname,
    "totalamount" : orders.totalamount
}
cust_orders = pd.DataFrame(cust_orders)


In [103]:
fig = px.bar(cust_orders.groupby('fullname')['totalamount'].count().sort_values(ascending=False).head(10), text_auto=True, title="Number of orders per customer",
labels={'fullname' : "Full Name", "value" : "Frequency"})
fig.update_traces(marker_color="skyblue", marker_line_color='grey', marker_line_width=1.5, opacity=0.5, textposition='outside')
fig.show()

In [105]:
fig = px.bar(cust_orders.groupby('fullname')['totalamount'].sum().sort_values(ascending=False).head(10), text_auto=True, title="Number of orders per customer",
labels={'fullname' : "Full Name", "value" : "Total Amount"})
fig.update_traces(marker_color="skyblue", marker_line_color='grey', marker_line_width=1.5, opacity=0.5, textposition='outside')
fig.show()